<a href="https://colab.research.google.com/github/vpfjt567/Skin_Care_team_project/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os

# 데이터 경로 설정
DATA_PATH = "/content/drive/MyDrive/SkinCare/data"

TRAIN_PATH = os.path.join(DATA_PATH, "train")
VAL_PATH = os.path.join(DATA_PATH, "val")

print("Training 폴더 파일 개수:", len(os.listdir(TRAIN_PATH)))
print("Validation 폴더 파일 개수:", len(os.listdir(VAL_PATH)))


Training 폴더 파일 개수: 15
Validation 폴더 파일 개수: 15


In [4]:
# 모델 기본 설정
MODEL_NAME = "SkinCare_MobileNetV3_Model"

EPOCHS = 10
IMAGE_SIZE = 224
BATCH_SIZE = 16
LEARNING_RATE = 0.01

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능: True
GPU 이름: NVIDIA L4


In [6]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 학습 데이터 전처리 및 증강 (피부 병변 특징 반영)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 검증 데이터 전처리 (증강 적용하지 않음)
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 학습 및 검증 데이터셋 생성
train_dataset = datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform)
val_dataset = datasets.ImageFolder(root=VAL_PATH, transform=val_transform)

# 클래스 확인
print("클래스 수:", len(train_dataset.classes))
print("클래스 목록:", train_dataset.classes)
print("Training 이미지:", len(train_dataset))
print("Validation 이미지:", len(val_dataset))
print("클래스 순서 일치:", train_dataset.class_to_idx == val_dataset.class_to_idx)


클래스 수: 15
클래스 목록: ['Actinic_Keratosis', 'Basal_Cell_Carcinoma', 'Bowen_disease', 'Dermatofibroma', 'Lentigo', 'Malignant_melanoma', 'Melanocytic_nevus', 'Milium', 'Pyogenic_granuloma', 'Sebaceous_hyperplasia', 'Seborrheic_keratosis', 'Squamous_Cell_Carcinoma', 'epidermal_cyst', 'hemangioma', 'verruca']
Training 이미지: 12000
Validation 이미지: 1500
클래스 순서 일치: True


In [7]:
from collections import Counter

class_counts = Counter(train_dataset.targets)

for i, name in enumerate(train_dataset.classes):
    print(f"{name}: {class_counts[i]}장")

Actinic_Keratosis: 800장
Basal_Cell_Carcinoma: 800장
Bowen_disease: 800장
Dermatofibroma: 800장
Lentigo: 800장
Malignant_melanoma: 800장
Melanocytic_nevus: 800장
Milium: 800장
Pyogenic_granuloma: 800장
Sebaceous_hyperplasia: 800장
Seborrheic_keratosis: 800장
Squamous_Cell_Carcinoma: 800장
epidermal_cyst: 800장
hemangioma: 800장
verruca: 800장


In [8]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training 배치 수:", len(train_loader))
print("Validation 배치 수:", len(val_loader))

Training 배치 수: 750
Validation 배치 수: 94


In [9]:
images, labels = next(iter(train_loader))

print("이미지 크기:", images.shape)
print("레이블 크기:", labels.shape)
print("GPU 이동 전:", images.device)

이미지 크기: torch.Size([16, 3, 224, 224])
레이블 크기: torch.Size([16])
GPU 이동 전: cpu


In [10]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ImageNet 사전학습 MobileNetV3 불러오기
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

# 마지막 분류층 변경
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, len(train_dataset.classes))

# 모델 전체를 학습 가능하도록 설정
for param in model.parameters():
    param.requires_grad = True

# GPU로 모델 이동
model = model.to(device)

print("사용 모델: MobileNetV3-Large")
print("사용 장치:", device)
print("분류 클래스:", model.classifier[3].out_features)
print("학습 가능 파라미터:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 214MB/s]

사용 모델: MobileNetV3-Large
사용 장치: cuda
분류 클래스: 15
학습 가능 파라미터: 4221247


In [11]:
import torch.optim as optim

# 손실 함수
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

# 학습 설정
EPOCHS = 10

print("Loss Function:", criterion)
print("Optimizer:", optimizer)
print("Learning Rate:", optimizer.param_groups[0]["lr"])
print("Epochs:", EPOCHS)


Loss Function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)
Learning Rate: 0.01
Epochs: 10


In [ ]:
import time
import copy
from tqdm.auto import tqdm

# 최고 성능 모델 기록
best_val_acc = 0.0
history = []

# 개선 모델 저장을 위한 별도 파일명
MODEL_NAME = "SkinCare_MobileNetV3_LS_Model"

# Google Drive에 모델 저장
SAVE_DIR = "/content/drive/MyDrive/Skin_Care_team_project/CNN_models"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}.pth")

start_time = time.time()

for epoch in range(EPOCHS):

    # =========================
    # 1. Training
    # =========================
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # =========================
    # 2. Validation
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")

        for images, labels in val_bar:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total
    scheduler.step(val_loss)

    # =========================
    # 3. 결과 출력 및 저장
    # =========================
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    # Validation Accuracy가 가장 높은 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "classes": train_dataset.classes
        }, SAVE_PATH)

        print(f"Best model 저장 완료! ({best_val_acc:.2f}%)")

# =========================
# 4. 전체 학습 시간
# =========================
total_time = time.time() - start_time

print("\n===== 학습 완료 =====")
print(f"총 학습 시간: {total_time / 60:.2f}분")
print(f"최고 Validation Accuracy: {best_val_acc:.2f}%")
print(f"모델 저장 위치: {SAVE_PATH}")

Epoch 1/10 [Train]:   0%|          | 0/750 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/94 [00:00<?, ?it/s]

In [ ]:
# 최고 성능 모델 불러오기
checkpoint = torch.load(
    SAVE_PATH,
    map_location=device,
    weights_only=False
)

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. 방금 저장한 98.33% 최고 모델 정확히 불러오기
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred_orig, y_pred_tta = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        # 원본 예측
        p_orig = torch.softmax(model(images), dim=1)
        _, pred_orig = torch.max(p_orig, 1)

        # 3-방향 TTA (원본 + 좌우 반전 + 상하 반전)
        p_hflip = torch.softmax(model(torch.flip(images, dims=[3])), dim=1)
        p_vflip = torch.softmax(model(torch.flip(images, dims=[2])), dim=1)
        probs_tta = (p_orig + p_hflip + p_vflip) / 3.0
        _, pred_tta = torch.max(probs_tta, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred_orig.extend(pred_orig.cpu().numpy())
        y_pred_tta.extend(pred_tta.cpu().numpy())

print(f"기본 최고 정확도: {accuracy_score(y_true, y_pred_orig)*100:.2f}%")
print(f"3-방향 TTA 최종 정확도: {accuracy_score(y_true, y_pred_tta)*100:.2f}%")
print(f"TTA Precision (Macro): {precision_score(y_true, y_pred_tta, average='macro'):.4f}")
print(f"TTA Recall (Macro): {recall_score(y_true, y_pred_tta, average='macro'):.4f}")
print(f"TTA F1-score (Macro): {f1_score(y_true, y_pred_tta, average='macro'):.4f}")


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. 저장된 두 모델 파일 경로 설정
path_model1 = os.path.join(SAVE_DIR, "SkinCare_MobileNetV3_Improved_Model.pth")  # 1차 개선 모델 (95.80%)
path_model2 = os.path.join(SAVE_DIR, "SkinCare_MobileNetV3_LS_Model.pth")        # 2차 라벨스무딩 모델 (98.33%)

def load_saved_model(path):
    m = mobilenet_v3_large(weights=None)
    # 기존 모델 구조(classifier[3] 교체)와 동일하게 맞춤
    num_features = m.classifier[3].in_features
    m.classifier[3] = nn.Linear(num_features, len(train_dataset.classes))

    ckpt = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt["model_state_dict"])
    m = m.to(device)
    m.eval()
    return m

model1 = load_saved_model(path_model1)
model2 = load_saved_model(path_model2)

# 2. 3-방향 TTA 확률 계산 함수
def get_tta_probs(m, imgs):
    p_orig = torch.softmax(m(imgs), dim=1)
    p_hflip = torch.softmax(m(torch.flip(imgs, dims=[3])), dim=1)
    p_vflip = torch.softmax(m(torch.flip(imgs, dims=[2])), dim=1)
    return (p_orig + p_hflip + p_vflip) / 3.0

y_true, y_pred_ens = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        # 두 모델의 TTA 예측 확률 결합 (성능이 높은 2번 모델 70% + 1번 모델 30%)
        probs1 = get_tta_probs(model1, images)
        probs2 = get_tta_probs(model2, images)
        final_probs = 0.3 * probs1 + 0.7 * probs2

        _, preds = torch.max(final_probs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred_ens.extend(preds.cpu().numpy())

print("===== 2개 모델 앙상블 + TTA 최종 결과 =====")
print(f"앙상블 최종 정확도: {accuracy_score(y_true, y_pred_ens)*100:.2f}%")
print(f"Precision (Macro): {precision_score(y_true, y_pred_ens, average='macro'):.4f}")
print(f"Recall (Macro): {recall_score(y_true, y_pred_ens, average='macro'):.4f}")
print(f"F1-score (Macro): {f1_score(y_true, y_pred_ens, average='macro'):.4f}")


In [ ]:
import os

print("모델 파일 존재:", os.path.isfile(SAVE_PATH))

if os.path.isfile(SAVE_PATH):
    print("파일 크기:", round(os.path.getsize(SAVE_PATH) / (1024 * 1024), 2), "MB")
    print("저장 경로:", SAVE_PATH)

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history)

display(history_df)

In [ ]:
import matplotlib.pyplot as plt

epochs = [item["epoch"] for item in history]
train_acc = [item["train_acc"] for item in history]
val_acc = [item["val_acc"] for item in history]
train_loss = [item["train_loss"] for item in history]
val_loss = [item["val_loss"] for item in history]

plt.figure(figsize=(14, 5))

# Accuracy 그래프
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc, marker='o', label='Train Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title('MobileNetV3 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.xticks(epochs)
plt.legend()
plt.grid(True)

# Loss 그래프
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss, marker='o', label='Train Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title('MobileNetV3 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(epochs)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 최고 성능 모델 불러오기
checkpoint = torch.load(
    SAVE_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("불러온 모델 Epoch:", checkpoint["epoch"])
print("최고 Validation Accuracy:", checkpoint["best_val_acc"])

In [ ]:
y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

print("예측 완료")
print("평가 이미지:", len(y_true))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 13))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.title("MobileNetV3 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
print("===== MobileNetV3 Evaluation =====")

print("Accuracy:",
      accuracy_score(y_true, y_pred))

print("Precision (Macro):",
      precision_score(y_true, y_pred, average="macro", zero_division=0))

print("Recall (Macro):",
      recall_score(y_true, y_pred, average="macro", zero_division=0))

print("F1-score (Macro):",
      f1_score(y_true, y_pred, average="macro", zero_division=0))

print("\n===== Classification Report =====")

print(classification_report(
    y_true,
    y_pred,
    target_names=train_dataset.classes,
    zero_division=0
))

In [ ]:
import os
import hashlib
from collections import Counter

def get_image_hashes(folder):
    hashes = []

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(root, file)

                with open(path, "rb") as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()

                hashes.append((file_hash, path))

    return hashes


train_hashes = get_image_hashes(TRAIN_PATH)
val_hashes = get_image_hashes(VAL_PATH)

train_hash_set = set(h for h, p in train_hashes)
val_hash_set = set(h for h, p in val_hashes)

# Training과 Validation 사이의 중복
duplicate_hashes = train_hash_set & val_hash_set

print("Training 이미지:", len(train_hashes))
print("Validation 이미지:", len(val_hashes))
print("Training 내부 고유 이미지:", len(train_hash_set))
print("Validation 내부 고유 이미지:", len(val_hash_set))
print("Training-Validation 중복 이미지:", len(duplicate_hashes))

In [ ]:
from collections import defaultdict
import os

train_hash_map = defaultdict(list)
val_hash_map = defaultdict(list)

for h, path in train_hashes:
    train_hash_map[h].append(path)

for h, path in val_hashes:
    val_hash_map[h].append(path)

print("===== Training 내부 중복 =====")

for h, paths in train_hash_map.items():
    if len(paths) > 1:
        print(f"\n중복 수: {len(paths)}")
        print(*paths, sep="\n")

print("\n===== Validation 내부 중복 =====")

for h, paths in val_hash_map.items():
    if len(paths) > 1:
        print(f"\n중복 수: {len(paths)}")
        print(*paths, sep="\n")

print("\n===== Training-Validation 중복 =====")

for h in train_hash_set & val_hash_set:
    print("\nTraining:", train_hash_map[h])
    print("Validation:", val_hash_map[h])

In [ ]:
print("데이터 개수:", len(train_dataset))
print("질환 종류:", train_dataset.classes)
print("첫 번째 이미지:", train_dataset.samples[0])